# Python, очистка данных и SQL в сквозном аналитическом кейсе

Практическая работа для слушателя.

Маршрут работы:

```text
получить данные -> загрузить -> проверить качество -> очистить -> рассчитать показатели -> сверить через SQL -> сделать вывод
```

Работайте сверху вниз. Не пропускайте ячейки: следующие блоки используют переменные, созданные выше.

## 0. Ожидаемые результаты

После выполнения notebook у вас будут:

| Результат | Переменная / папка |
|---|---|
| Загруженные таблицы | `sales`, `products`, `regions`, `clients` |
| Очищенная таблица продаж | `sales_clean` |
| Полная таблица для анализа | `sales_full` |
| Итог по регионам | `region_summary` |
| Итог по категориям | `category_summary` |
| Сводная таблица категорий и каналов | `category_channel_pivot` |
| SQL-сверка | `sql_region_summary` |
| Файлы результата | `outputs/` |

## 1. Подготовка рабочей среды

Подключаем библиотеки и находим корень проекта. Корень проекта — папка, где лежат `data`, `sql`, `outputs`, `notebooks`.

In [ ]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

In [ ]:
def find_project_root(start_path=None):
    # Ищет папку проекта по наличию data/raw/sales.csv.
    start = Path.cwd() if start_path is None else Path(start_path)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "raw" / "sales.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Не удалось найти корень проекта. Проверьте, что внутри проекта есть файл data/raw/sales.csv"
    )

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
SQL_DIR = PROJECT_ROOT / "sql"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Корень проекта:", PROJECT_ROOT)
print("Папка исходных данных:", RAW_DIR)
print("Папка обработанных данных:", PROCESSED_DIR)
print("Папка результатов:", OUTPUT_DIR)
print("Папка SQL:", SQL_DIR)

## Русские подписи для таблиц

В коде ниже мы создаём словарь русских названий столбцов и функцию `display_ru()`. Она нужна только для удобного чтения результата. Исходные таблицы и переменные не переименовываются, поэтому код и SQL-запросы продолжают работать как раньше.


In [ ]:
RUSSIAN_COLUMNS = {
    "order_id": "Номер заказа",
    "order_date": "Дата заказа",
    "client_id": "Код клиента",
    "product_id": "Код товара",
    "region_id": "Код региона",
    "channel": "Канал продаж",
    "quantity": "Количество",
    "unit_price": "Цена за единицу",
    "discount": "Скидка",
    "revenue": "Выручка",
    "order_month": "Месяц заказа",
    "is_discounted": "Есть скидка",
    "product_name": "Название товара",
    "category": "Категория",
    "cost": "Себестоимость",
    "region_name": "Регион",
    "macro_region": "Макрорегион",
    "segment": "Сегмент клиента",
    "registration_date": "Дата регистрации",
    "loyalty_level": "Уровень лояльности",
    "orders_count": "Количество заказов",
    "clients_count": "Количество клиентов",
    "total_quantity": "Общее количество",
    "total_revenue": "Общая выручка",
    "avg_order_revenue": "Средняя выручка на заказ",
    "avg_discount": "Средняя скидка",
    "discount_group": "Группа скидки",
    "revenue_diff": "Разница в выручке",
    "orders_diff": "Разница в заказах",
    "problem": "Проблема",
    "rows_count": "Количество строк",
}

RUSSIAN_PROBLEMS = {
    "orders_without_product": "заказы без найденного товара",
    "orders_without_region": "заказы без найденного региона",
    "orders_without_client": "заказы без найденного клиента",
}

def display_ru(df, rows=10):
    """Показать таблицу с русскими подписями столбцов, не меняя исходный DataFrame."""
    ru_df = df.copy()
    if "problem" in ru_df.columns:
        ru_df["problem"] = ru_df["problem"].replace(RUSSIAN_PROBLEMS)
    ru_df = ru_df.rename(columns=RUSSIAN_COLUMNS)
    display(ru_df.head(rows))
    return ru_df

# Проверка: функция готова к использованию.
print("Русские подписи для таблиц подключены.")


### 1.1. Проверка файлов

Перед загрузкой данных проверяем, что все файлы лежат на месте.

In [ ]:
required_files = {
    "sales": RAW_DIR / "sales.csv",
    "products": RAW_DIR / "products.xlsx",
    "regions": RAW_DIR / "regions.json",
    "clients": RAW_DIR / "clients.csv",
    "sqlite_db": SQL_DIR / "analytics_demo.sqlite",
}

for file_name, file_path in required_files.items():
    status = "OK" if file_path.exists() else "НЕ НАЙДЕН"
    print(f"{status:10} {file_name:10} {file_path}")

**Контрольная точка:** напротив каждого файла должно быть `OK`. Если файл не найден, проверьте структуру папок и название файла.

## 2. Загрузка данных

Загружаем четыре исходные таблицы.

In [ ]:
sales = pd.read_csv(RAW_DIR / "sales.csv")
products = pd.read_excel(RAW_DIR / "products.xlsx")
regions = pd.read_json(RAW_DIR / "regions.json")
clients = pd.read_csv(RAW_DIR / "clients.csv")

print("sales:", sales.shape)
print("products:", products.shape)
print("regions:", regions.shape)
print("clients:", clients.shape)

In [ ]:
display(sales.head())
display(products.head())
display(regions.head())
display(clients.head())

## 3. Первичная диагностика качества

Перед расчётами проверяем размер, типы данных, пропуски, дубликаты и бизнес-ошибки.

In [ ]:
print("Размер sales:", sales.shape)
sales.info()

In [ ]:
print("Типы данных в sales:")
display(sales.dtypes)

### 3.1. Пропуски

In [ ]:
missing_sales = sales.isna().sum().sort_values(ascending=False)
display(missing_sales)

### 3.2. Дубликаты заказов

Если один и тот же `order_id` встречается несколько раз, итоговая выручка может быть завышена.

In [ ]:
duplicate_orders = sales["order_id"].duplicated().sum()
print("Количество дубликатов order_id:", duplicate_orders)

if duplicate_orders > 0:
    display(sales[sales["order_id"].duplicated(keep=False)].sort_values("order_id").head(10))

### 3.3. Бизнес-ошибки

Проверяем невозможные или подозрительные значения: отрицательное количество, нулевая цена, скидка вне диапазона 0–1.

In [ ]:
checks = {
    "negative_quantity": sales["quantity"] < 0,
    "zero_or_negative_unit_price": sales["unit_price"] <= 0,
    "bad_discount": (sales["discount"] < 0) | (sales["discount"] > 1),
}

for check_name, mask in checks.items():
    print(f"{check_name}: {mask.sum()}")
    if mask.sum() > 0:
        display(sales.loc[mask].head())

### 3.4. Текстовые поля

Для человека `Online`, `online` и ` online ` похожи, но для программы это разные строки.

In [ ]:
print("Каналы продаж до очистки:")
display(sales["channel"].value_counts(dropna=False))

print("Категории товаров до очистки:")
display(products["category"].value_counts(dropna=False))

### 3.5. Уникальность ключей в справочниках

Ключ в справочнике должен быть уникальным: один `product_id` — один товар.

In [ ]:
print("Дубликаты product_id в products:", products["product_id"].duplicated().sum())
print("Дубликаты region_id в regions:", regions["region_id"].duplicated().sum())
print("Дубликаты client_id в clients:", clients["client_id"].duplicated().sum())

if products["product_id"].duplicated().sum() > 0:
    display(products[products["product_id"].duplicated(keep=False)].sort_values("product_id"))

## 4. Очистка данных

Создаём отдельные копии таблиц для очистки. Исходные таблицы оставляем без изменений.

In [ ]:
sales_clean = sales.copy()
products_clean = products.copy()
regions_clean = regions.copy()
clients_clean = clients.copy()

### 4.1. Преобразование дат

Дата из файла часто загружается как текст. Для анализа по времени её нужно преобразовать в тип даты.

In [ ]:
sales_clean["order_date"] = pd.to_datetime(sales_clean["order_date"], errors="coerce")
clients_clean["registration_date"] = pd.to_datetime(clients_clean["registration_date"], errors="coerce")

print("Тип order_date:", sales_clean["order_date"].dtype)
print("Нераспознанные даты заказов:", sales_clean["order_date"].isna().sum())
print("Нераспознанные даты регистрации:", clients_clean["registration_date"].isna().sum())

### 4.2. Очистка текстовых полей

Убираем лишние пробелы и приводим значения к единому виду.

In [ ]:
sales_clean["channel"] = sales_clean["channel"].str.strip().str.lower()
products_clean["category"] = products_clean["category"].str.strip().str.lower()
clients_clean["segment"] = clients_clean["segment"].str.strip()
clients_clean["loyalty_level"] = clients_clean["loyalty_level"].str.strip().str.title()

print("Каналы после очистки:")
display(sales_clean["channel"].value_counts(dropna=False))

print("Категории после очистки:")
display(products_clean["category"].value_counts(dropna=False))

### 4.3. Удаление дубликатов заказов

В рамках учебного кейса считаем, что дубли `order_id` — технические дубли. Оставляем первую строку по каждому заказу.

In [ ]:
rows_before = len(sales_clean)
sales_clean = sales_clean.drop_duplicates(subset=["order_id"], keep="first")
rows_after = len(sales_clean)

print("Строк до удаления дублей:", rows_before)
print("Строк после удаления дублей:", rows_after)
print("Удалено строк:", rows_before - rows_after)
print("Дубликаты order_id после очистки:", sales_clean["order_id"].duplicated().sum())

### 4.4. Фильтрация некорректных числовых значений

Оставляем только строки, которые проходят базовые бизнес-правила:

- `quantity > 0`;
- `unit_price > 0`;
- `discount` от 0 до 1.

In [ ]:
valid_sales_mask = (
    (sales_clean["quantity"] > 0) &
    (sales_clean["unit_price"] > 0) &
    (sales_clean["discount"].between(0, 1))
)

invalid_sales = sales_clean.loc[~valid_sales_mask].copy()
print("Некорректных строк будет исключено:", len(invalid_sales))
display(invalid_sales.head())

sales_clean = sales_clean.loc[valid_sales_mask].copy()
print("Размер sales_clean после фильтрации:", sales_clean.shape)

### 4.5. Очистка справочника товаров

Если в справочнике товаров есть дубликаты `product_id`, при объединении продажи могут размножиться.

In [ ]:
products_before = len(products_clean)
products_clean = products_clean.drop_duplicates(subset=["product_id"], keep="first")
products_after = len(products_clean)

print("Строк в products до удаления дублей:", products_before)
print("Строк в products после удаления дублей:", products_after)
print("Дубликаты product_id после очистки:", products_clean["product_id"].duplicated().sum())

### 4.6. Повторная проверка после очистки

In [ ]:
print("Размер sales_clean:", sales_clean.shape)
print("Пропуски в sales_clean:")
display(sales_clean.isna().sum())

print("Проверка бизнес-правил после очистки:")
print("quantity <= 0:", (sales_clean["quantity"] <= 0).sum())
print("unit_price <= 0:", (sales_clean["unit_price"] <= 0).sum())
print("discount вне диапазона 0-1:", ((sales_clean["discount"] < 0) | (sales_clean["discount"] > 1)).sum())

## 5. Расчёт показателей в pandas

Теперь данные готовы для базовых аналитических расчётов.

### 5.1. Расчёт выручки

Формула:

```text
revenue = quantity * unit_price * (1 - discount)
```

In [ ]:
sales_clean["revenue"] = sales_clean["quantity"] * sales_clean["unit_price"] * (1 - sales_clean["discount"])
sales_clean["order_month"] = sales_clean["order_date"].dt.to_period("M").astype(str)
sales_clean["is_discounted"] = sales_clean["discount"] > 0

display(sales_clean[["order_id", "quantity", "unit_price", "discount", "revenue", "order_month", "is_discounted"]].head())

### 5.2. Объединение продаж со справочниками

Подтягиваем к продажам названия товаров, категории, регионы и клиентские сегменты.

In [ ]:
sales_products = sales_clean.merge(products_clean, on="product_id", how="left")
sales_full = sales_products.merge(regions_clean, on="region_id", how="left")
sales_full = sales_full.merge(clients_clean, on="client_id", how="left")

print("Строк в sales_clean:", len(sales_clean))
print("Строк после объединения:", len(sales_full))
display(sales_full.head())

In [ ]:
merge_quality = pd.DataFrame({
    "problem": ["orders_without_product", "orders_without_region", "orders_without_client"],
    "rows_count": [
        sales_full["product_name"].isna().sum(),
        sales_full["region_name"].isna().sum(),
        sales_full["segment"].isna().sum(),
    ]
})

display(merge_quality)

In [ ]:
display_ru(merge_quality) # русские подписи для проверки объединения

Если после объединения появились пропуски в `product_name`, `region_name` или `segment`, значит часть ключей не нашлась в справочниках.

### 5.3. Выручка по регионам

In [ ]:
region_summary = (
    sales_full
    .groupby("region_name", dropna=False, as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        total_revenue=("revenue", "sum"),
        avg_order_revenue=("revenue", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)

region_summary["total_revenue"] = region_summary["total_revenue"].round(2)
region_summary["avg_order_revenue"] = region_summary["avg_order_revenue"].round(2)

display(region_summary)

In [ ]:
display_ru(region_summary) # та же таблица с русскими подписями

### 5.4. Выручка по категориям

In [ ]:
category_summary = (
    sales_full
    .groupby("category", dropna=False, as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
        avg_discount=("discount", "mean"),
    )
    .sort_values("total_revenue", ascending=False)
)

category_summary["total_revenue"] = category_summary["total_revenue"].round(2)
category_summary["avg_discount"] = category_summary["avg_discount"].round(3)

display(category_summary)

In [ ]:
display_ru(category_summary) # та же таблица с русскими подписями

### 5.5. Сводная таблица: категории × каналы продаж

In [ ]:
category_channel_pivot = pd.pivot_table(
    sales_full,
    index="category",
    columns="channel",
    values="revenue",
    aggfunc="sum",
    fill_value=0,
)

category_channel_pivot = category_channel_pivot.round(2)
display(category_channel_pivot)

### 5.6. Выручка по месяцам

In [ ]:
monthly_summary = (
    sales_full
    .groupby("order_month", dropna=False, as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
    )
    .sort_values("order_month")
)

monthly_summary["total_revenue"] = monthly_summary["total_revenue"].round(2)
display(monthly_summary)

In [ ]:
display_ru(monthly_summary) # та же таблица с русскими подписями

## 6. Сохранение результатов

Сохраняем очищенные данные и итоговые таблицы в папку `outputs`.

In [ ]:
sales_clean.to_csv(OUTPUT_DIR / "sales_clean.csv", index=False)
sales_full.to_csv(OUTPUT_DIR / "sales_full.csv", index=False)
region_summary.to_csv(OUTPUT_DIR / "region_summary.csv", index=False)
category_summary.to_csv(OUTPUT_DIR / "category_summary.csv", index=False)
category_channel_pivot.to_excel(OUTPUT_DIR / "category_channel_pivot.xlsx")
monthly_summary.to_csv(OUTPUT_DIR / "monthly_summary.csv", index=False)

print("Сохранённые файлы:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    print(file_path.name)

## 7. SQL-блок через SQLite

Сначала подключимся к готовой SQLite-базе и выполним несколько базовых запросов.

In [ ]:
db_path = SQL_DIR / "analytics_demo.sqlite"
connection = sqlite3.connect(db_path)

print("Подключение к базе создано:", db_path)

In [ ]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    connection,
)
display(tables)

### 7.1. Первый SELECT

In [ ]:
query = """
SELECT *
FROM sales
LIMIT 5;
"""

pd.read_sql_query(query, connection)

### 7.2. Фильтрация и сортировка

In [ ]:
query = """
SELECT
    order_id,
    order_date,
    region_id,
    channel,
    quantity,
    unit_price,
    discount
FROM sales
WHERE quantity > 0
ORDER BY order_date
LIMIT 10;
"""

pd.read_sql_query(query, connection)

### 7.3. Агрегация по регионам в SQL на исходной базе

Этот запрос работает с исходной SQLite-базой. Он фильтрует некорректные значения, но не повторяет всю pandas-очистку, например удаление дублей. Поэтому итог может отличаться от `region_summary`.

In [ ]:
query = """
SELECT
    region_id,
    COUNT(DISTINCT order_id) AS orders_count,
    SUM(quantity * unit_price * (1 - discount)) AS total_revenue,
    AVG(quantity * unit_price * (1 - discount)) AS avg_order_revenue
FROM sales
WHERE quantity > 0
  AND unit_price > 0
  AND discount BETWEEN 0 AND 1
GROUP BY region_id
ORDER BY total_revenue DESC;
"""

sql_raw_region_summary = pd.read_sql_query(query, connection)
display(sql_raw_region_summary)

### 7.4. JOIN в SQL

In [ ]:
query = """
SELECT
    r.region_name,
    p.category,
    COUNT(DISTINCT s.order_id) AS orders_count,
    SUM(s.quantity * s.unit_price * (1 - s.discount)) AS total_revenue
FROM sales AS s
LEFT JOIN products AS p
    ON s.product_id = p.product_id
LEFT JOIN regions AS r
    ON s.region_id = r.region_id
WHERE s.quantity > 0
  AND s.unit_price > 0
  AND s.discount BETWEEN 0 AND 1
GROUP BY r.region_name, p.category
ORDER BY total_revenue DESC
LIMIT 20;
"""

sql_raw_category_region = pd.read_sql_query(query, connection)
display(sql_raw_category_region)

### 7.5. SQL-сверка по очищенным данным

Чтобы честно сравнить pandas и SQL, нужно использовать одинаково очищенные данные. Создадим временную SQLite-базу в памяти и загрузим туда очищенную таблицу.

In [ ]:
clean_connection = sqlite3.connect(":memory:")
sales_full.to_sql("sales_full", clean_connection, index=False, if_exists="replace")

query = """
SELECT
    region_name,
    COUNT(DISTINCT order_id) AS orders_count,
    SUM(revenue) AS total_revenue,
    AVG(revenue) AS avg_order_revenue
FROM sales_full
GROUP BY region_name
ORDER BY total_revenue DESC;
"""

sql_region_summary = pd.read_sql_query(query, clean_connection)
sql_region_summary["total_revenue"] = sql_region_summary["total_revenue"].round(2)
sql_region_summary["avg_order_revenue"] = sql_region_summary["avg_order_revenue"].round(2)

display(sql_region_summary)

In [ ]:
display_ru(sql_region_summary) # SQL-результат с русскими подписями

## 8. Сравнение pandas и SQL

Сравним `region_summary`, полученный в pandas, и `sql_region_summary`, полученный через SQL по очищенной таблице.

In [ ]:
pandas_compare = region_summary[["region_name", "orders_count", "total_revenue", "avg_order_revenue"]].copy()
sql_compare = sql_region_summary[["region_name", "orders_count", "total_revenue", "avg_order_revenue"]].copy()

comparison = pandas_compare.merge(
    sql_compare,
    on="region_name",
    how="outer",
    suffixes=("_pandas", "_sql"),
)

comparison["revenue_diff"] = (comparison["total_revenue_pandas"] - comparison["total_revenue_sql"]).round(2)
comparison["orders_diff"] = comparison["orders_count_pandas"] - comparison["orders_count_sql"]

display(comparison)

In [ ]:
display_ru(comparison) # сравнение pandas и SQL с русскими подписями

In [ ]:
print("Максимальная разница по выручке:", comparison["revenue_diff"].abs().max())
print("Максимальная разница по количеству заказов:", comparison["orders_diff"].abs().max())

**Вывод для проверки:** если pandas и SQL работают по одной и той же очищенной таблице и по одной формуле, разница должна быть нулевой или объяснимой округлением.

## 9. Самостоятельное мини-задание

### Задание 1

Посчитайте выручку по сегментам клиентов (`segment`).

In [ ]:
segment_summary = (
    sales_full
    .groupby("segment", dropna=False, as_index=False)
    .agg(
        orders_count=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
    )
    .sort_values("total_revenue", ascending=False)
)

segment_summary["total_revenue"] = segment_summary["total_revenue"].round(2)
display(segment_summary)

In [ ]:
display_ru(segment_summary) # итог по сегментам с русскими подписями

### Задание 2

Посчитайте выручку по уровням лояльности (`loyalty_level`).

In [ ]:
loyalty_summary = (
    sales_full
    .groupby("loyalty_level", dropna=False, as_index=False)
    .agg(
        clients_count=("client_id", "nunique"),
        orders_count=("order_id", "nunique"),
        total_revenue=("revenue", "sum"),
    )
    .sort_values("total_revenue", ascending=False)
)

loyalty_summary["total_revenue"] = loyalty_summary["total_revenue"].round(2)
display(loyalty_summary)

In [ ]:
display_ru(loyalty_summary) # итог по лояльности с русскими подписями

### Задание 3

Сохраните дополнительные таблицы в `outputs`.

In [ ]:
segment_summary.to_csv(OUTPUT_DIR / "segment_summary.csv", index=False)
loyalty_summary.to_csv(OUTPUT_DIR / "loyalty_summary.csv", index=False)

print("Дополнительные файлы сохранены.")

## 10. Итоговый аналитический вывод

Заполните вывод своими словами.

Структура вывода:

```text
1. Что анализировалось:
2. Какие проблемы качества данных были найдены:
3. Какие действия по очистке были выполнены:
4. Какие показатели рассчитаны:
5. Какие регионы / категории / каналы показали высокий результат:
6. Совпали ли результаты pandas и SQL:
7. Какие ограничения анализа нужно учитывать:
```

### Мой вывод

> ...

## 11. Финальная самопроверка

| Проверка | Да / нет |
|---|---|
| Все исходные файлы найдены |  |
| Данные загружены |  |
| Проверены пропуски |  |
| Проверены дубликаты |  |
| Даты преобразованы |  |
| Текстовые поля очищены |  |
| Некорректные строки исключены |  |
| Справочник товаров очищен от дублей |  |
| Рассчитана выручка |  |
| Выполнено объединение таблиц |  |
| Созданы итоговые таблицы |  |
| Результаты сохранены в `outputs` |  |
| Выполнены SQL-запросы |  |
| Выполнена сверка pandas и SQL |  |
| Написан аналитический вывод |  |

In [ ]:
connection.close()
clean_connection.close()
print("Соединения с SQLite закрыты. Практическая работа завершена.")

### Подсказка по русским формулировкам

В итоговом выводе используйте русские названия показателей: «общая выручка», «количество заказов», «средняя выручка на заказ», «регион», «категория», «канал продаж». Технические имена вроде `total_revenue` и `orders_count` можно оставить только в коде.
